# TripAdvisor User Ratings Clustering Analysis
Since the TripAdvisor dataset contains numerical ratings across 10 categories (and no actual text reviews), we will apply K-Means Clustering and Principal Component Analysis (PCA) to discover user segmentation patterns.

## 1. Import Dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')


## 2. Load and Inspect Data

In [ ]:
df = pd.read_csv('tripadvisor_review.csv')
display(df.head())
print("Dataset Shape:", df.shape)


## 3. Data Preprocessing

In [ ]:
# Drop the 'User ID' as it is non-informative for clustering
X_original = df.drop('User ID', axis=1)

# Feature Scaling is crucial for K-Means Since ratings could have different variances
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_original)

# Keep scaled data in dataframe for easier viewing
X_scaled_df = pd.DataFrame(X_scaled, columns=X_original.columns)
display(X_scaled_df.head())


## 4. Find optimal number of clusters (Elbow Method)

In [ ]:
wcss = [] # Within-Cluster Sum of Squares
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, wcss, marker='o', linestyle='--')
plt.title('Elbow Method For Optimal K')
plt.xlabel('Number of clusters (K)')
plt.ylabel('WCSS')
plt.xticks(k_range)
plt.show()

print("The 'Elbow' typically appears where the bend is sharpest (e.g., K=3 or K=4).")


## 5. Build Final K-Means Model (k=3)

In [ ]:
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42)
clusters = kmeans.fit_predict(X_scaled)

# Add cluster labels to original df
df['Cluster'] = clusters
score = silhouette_score(X_scaled, clusters)
print("Silhouette Score for k=3:", round(score, 3))


## 6. Dimensionality Reduction & Visualization with PCA

In [ ]:
# Reduce 10 categories to 2 Dimensions for 2D Plotting
pca = PCA(n_components=2)
reduced_X = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
sns.scatterplot(x=reduced_X[:, 0], y=reduced_X[:, 1], hue=clusters, palette='viridis', style=clusters, s=100)

# Plotting the centroids
centroids_pca = pca.transform(kmeans.cluster_centers_)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], s=300, c='red', marker='X', label='Centroids')

plt.title('Clusters Visualized via PCA (2 Components)')
plt.xlabel(f'PCA Component 1 ({round(pca.explained_variance_ratio_[0]*100, 1)}% Variance)')
plt.ylabel(f'PCA Component 2 ({round(pca.explained_variance_ratio_[1]*100, 1)}% Variance)')
plt.legend()
plt.show()


## 7. Analyze Clusters

In [ ]:
# Calculate the average rating per category for each cluster to understand user segments
cluster_means = df.drop('User ID', axis=1).groupby('Cluster').mean()
plt.figure(figsize=(12, 6))
sns.heatmap(cluster_means.T, cmap='YlGnBu', annot=True, fmt=".2f")
plt.title("Average Category Ratings per Cluster")
plt.show()
